# MYSignVoice YOLO26s Version 2 fine-tuning

Run the numbered cells from top to bottom in RunPod JupyterLab or Google Colab. The notebook downloads the frozen Roboflow Version 2 dataset, verifies the exact deployed 63-class order and the 8,380-image split, measures Version 1 on the Version 2 validation set, fine-tunes from Version 1 best.pt, compares both models, and packages the result.

Before Cell 6, upload the existing Version 1 best.pt to /workspace/best.pt. Do not use resume=True when introducing the new dataset; this is a new fine-tuning run.

## Cell 1 - Install tested packages

Ultralytics is pinned to the Version 1 training version. pandas and matplotlib are included so report cells work on a fresh Pod.

In [ ]:
%pip install -q "ultralytics==8.4.128" roboflow pyyaml pandas matplotlib openvino onnx onnxruntime

## Cell 2 - Confirm GPU and versions

In [ ]:
import platform
import torch
import ultralytics

assert torch.cuda.is_available(), "No CUDA GPU detected. Use a GPU RunPod."
gpu = torch.cuda.get_device_properties(0)
print("Python:", platform.python_version())
print("Ultralytics:", ultralytics.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("VRAM (GB):", round(gpu.total_memory / 1024**3, 1))
!nvidia-smi

## Cell 3 - Set persistent storage and enter the Roboflow API key

RunPod work is stored under /workspace. The API key is requested privately and is not saved in the notebook.

In [ ]:
import os
from getpass import getpass
from pathlib import Path

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    BASE_DIR = Path("/content")
    BACKUP_ROOT = Path("/content/drive/MyDrive/TrafficSignProject/training_runs")
    api_key = userdata.get("ROBOFLOW_API_KEY")
else:
    BASE_DIR = Path("/workspace")
    BACKUP_ROOT = BASE_DIR / "mysignvoice_artifacts"
    api_key = os.environ.get("ROBOFLOW_API_KEY") or getpass("Roboflow private API key: ")

assert api_key, "ROBOFLOW_API_KEY is missing."
os.environ["ROBOFLOW_API_KEY"] = api_key
BACKUP_ROOT.mkdir(parents=True, exist_ok=True)
print("Platform:", "Google Colab" if IN_COLAB else "RunPod/JupyterLab")
print("Persistent output folder:", BACKUP_ROOT)

## Cell 4 - Version 2 configuration

Set VERSION to the number Roboflow assigns the new 8,380-image version. Upload this project's models/best.pt as /workspace/best.pt.

In [ ]:
WORKSPACE = "kendrewlim-yahoo-com"
PROJECT = "mysignvoice-49-signs"
VERSION = 2

BASE_MODEL_PATH = BASE_DIR / "best.pt"
EXPECTED_TOTAL_IMAGES = 8380
EXPECTED_SPLIT_IMAGES = {"train": 6030, "val": 1567, "test": 783}

EPOCHS = 100
IMAGE_SIZE = 640
BATCH_SIZE = 16
PATIENCE = 20
WORKERS = 8
CLASS_WEIGHT_POWER = 0.25

DATASET_DIR = BASE_DIR / f"mysignvoice_rf_v{VERSION}"
RUN_ROOT = BASE_DIR / "mysignvoice_runs"
RUN_NAME = f"yolo26s_63class_rf_v{VERSION}_finetune_v1"
RUN_DIR = RUN_ROOT / RUN_NAME
EVAL_ROOT = BASE_DIR / "mysignvoice_evaluations"
REPORT_DIR = BASE_DIR / f"mysignvoice_report_rf_v{VERSION}"

for folder in (RUN_ROOT, EVAL_ROOT, REPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("Roboflow version:", VERSION)
print("Base model:", BASE_MODEL_PATH)
print("Run name:", RUN_NAME)

## Cell 5 - Locked deployed 63-class ID order

The list comes from the Version 1 deployed model and must match both best.pt and the Version 2 export exactly.

In [ ]:
EXPECTED_CLASSES = [
    "accident-prone-area-warning", "bicycle-path", "bicycle-warning",
    "bumps-warning", "bus-stop", "camera-operation-zone", "cars-only",
    "chevron-left", "chevron-right", "children-crossing-warning",
    "construction-ahead-warning", "cow-nearby-warning",
    "crossroad-left-warning", "crossroad-right-warning",
    "gated-railway-crossing-ahead-warning", "general-warning", "give-way",
    "height-limit", "left-or-right", "left-turn-only", "no-cars",
    "no-entry", "no-horn", "no-left", "no-left-and-right",
    "no-overtaking", "no-parking", "no-right", "no-straight",
    "no-straight-or-left", "no-uturn", "parking-area",
    "pass-obstacle-on-either-side", "pass-right",
    "pedestrian-crossing-warning", "railway-crossing-ahead-warning",
    "reverse-turn-warning", "right-turn-only", "road-narrows-left-warning",
    "road-narrows-right-warning", "roadway-diverges-warning", "roundabout",
    "sharp-right-turn-warning", "slippery-road-warning", "slowdown-warning",
    "speed-limit-15", "speed-limit-30", "speed-limit-40", "speed-limit-5",
    "speed-limit-50", "speed-limit-60", "speed-limit-80",
    "steep-descent-warning", "stop-for-inspection", "stop-sign",
    "straight-only", "straight-or-right", "towing-area",
    "traffic-light-ahead", "use-horn", "uturn-lane",
    "village-ahead-warning", "winding-road-warning",
]
assert len(EXPECTED_CLASSES) == 63 and len(set(EXPECTED_CLASSES)) == 63
print("Locked class count:", len(EXPECTED_CLASSES))

## Cell 6 - Verify uploaded Version 1 best.pt

Stop if this fails. Do not substitute generic yolo26s.pt.

In [ ]:
from ultralytics import YOLO

assert BASE_MODEL_PATH.is_file(), (
    f"Missing {BASE_MODEL_PATH}. Upload models/best.pt to this exact path."
)
base_model = YOLO(str(BASE_MODEL_PATH), task="detect")
base_names = base_model.names
if isinstance(base_names, dict):
    base_names = [base_names[i] for i in range(len(base_names))]
assert base_names == EXPECTED_CLASSES, (
    "The checkpoint is not the locked MYSignVoice Version 1 model or its class IDs differ."
)
print("PASS: Version 1 best.pt has the exact 63-class order.")

## Cell 7 - Download the frozen Roboflow version

Roboflow's YOLO26 export uses the Ultralytics detection layout required by YOLO26.

In [ ]:
from roboflow import Roboflow

existing_yaml = DATASET_DIR / "data.yaml"
if existing_yaml.is_file():
    DATASET_ROOT = DATASET_DIR
    print("Reusing existing dataset:", DATASET_ROOT)
else:
    rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
    rf_version = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION)
    dataset = rf_version.download(model_format="yolo26", location=str(DATASET_DIR))
    DATASET_ROOT = Path(dataset.location)

DATA_YAML = DATASET_ROOT / "data.yaml"
assert DATA_YAML.is_file(), f"Missing export configuration: {DATA_YAML}"
print("Dataset root:", DATASET_ROOT)
print("Dataset YAML:", DATA_YAML)

## Cell 8 - Validate class IDs, paths, splits, labels, and class balance

This safety gate stops if the dataset was rebalanced, the wrong version was downloaded, class IDs changed, or a label is invalid. Negative images may have no label file.

In [ ]:
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import yaml

with DATA_YAML.open("r", encoding="utf-8") as stream:
    data_config = yaml.safe_load(stream)

exported_names = data_config.get("names", [])
if isinstance(exported_names, dict):
    exported_names = [
        exported_names[i] if i in exported_names else exported_names[str(i)]
        for i in range(len(exported_names))
    ]
NAME_CORRECTIONS = {
    "pass-obstacles-on-either-side": "pass-obstacle-on-either-side",
    "winding_road_warning": "winding-road-warning",
}
corrected_names = [NAME_CORRECTIONS.get(name, name) for name in exported_names]
assert corrected_names == EXPECTED_CLASSES, (
    "Roboflow class names or IDs do not match the deployed model.\n"
    f"Expected: {EXPECTED_CLASSES}\nExported: {corrected_names}"
)

split_folders = {"train": "train", "val": "valid", "test": "test"}
split_image_dirs = {}
for yaml_key, folder_name in split_folders.items():
    raw_path = data_config.get(yaml_key)
    candidates = []
    if raw_path:
        raw_path = Path(raw_path)
        candidates.append(raw_path if raw_path.is_absolute() else (DATA_YAML.parent / raw_path).resolve())
    candidates.append((DATASET_ROOT / folder_name / "images").resolve())
    image_dir = next((path for path in candidates if path.is_dir()), None)
    assert image_dir is not None, f"Cannot find {yaml_key} images. Tried: {candidates}"
    split_image_dirs[yaml_key] = image_dir
    data_config[yaml_key] = str(image_dir)

data_config["names"] = EXPECTED_CLASSES
data_config["nc"] = len(EXPECTED_CLASSES)
with DATA_YAML.open("w", encoding="utf-8") as stream:
    yaml.safe_dump(data_config, stream, sort_keys=False, allow_unicode=True)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
split_rows = []
train_box_counts = Counter()
invalid_labels = []
for split_name, image_dir in split_image_dirs.items():
    label_dir = image_dir.parent / "labels"
    images = [p for p in image_dir.iterdir() if p.suffix.lower() in image_extensions]
    label_files = list(label_dir.glob("*.txt")) if label_dir.is_dir() else []
    box_count = 0
    for label_file in label_files:
        for line_number, line in enumerate(label_file.read_text(encoding="utf-8").splitlines(), 1):
            if not line.strip():
                continue
            parts = line.split()
            try:
                class_id = int(parts[0])
                coordinates = [float(value) for value in parts[1:]]
                assert len(coordinates) == 4
                assert 0 <= class_id < len(EXPECTED_CLASSES)
                assert all(0.0 <= value <= 1.0 for value in coordinates)
            except (ValueError, AssertionError):
                invalid_labels.append(f"{label_file}:{line_number}: {line}")
                continue
            box_count += 1
            if split_name == "train":
                train_box_counts[class_id] += 1
    split_rows.append({
        "split": split_name,
        "images": len(images),
        "label_files": len(label_files),
        "boxes": box_count,
        "negative_images": len(images) - len(label_files),
    })

assert not invalid_labels, "Invalid labels:\n" + "\n".join(invalid_labels[:20])
split_summary = pd.DataFrame(split_rows).set_index("split")
actual_splits = {name: int(split_summary.loc[name, "images"]) for name in EXPECTED_SPLIT_IMAGES}
assert actual_splits == EXPECTED_SPLIT_IMAGES, (
    "Dataset split mismatch. Do not train; the version may have been rebalanced. "
    f"Expected {EXPECTED_SPLIT_IMAGES}, got {actual_splits}."
)
assert sum(actual_splits.values()) == EXPECTED_TOTAL_IMAGES

class_distribution = pd.DataFrame({
    "class_id": range(len(EXPECTED_CLASSES)),
    "class_name": EXPECTED_CLASSES,
    "train_boxes": [train_box_counts[i] for i in range(len(EXPECTED_CLASSES))],
})
split_summary.to_csv(REPORT_DIR / "dataset_split_summary.csv")
class_distribution.to_csv(REPORT_DIR / "class_distribution.csv", index=False)
plt.figure(figsize=(12, 16))
ordered = class_distribution.sort_values("train_boxes")
plt.barh(ordered["class_name"], ordered["train_boxes"], color="#2563eb")
plt.xlabel("Training bounding boxes")
plt.title("MYSignVoice Version 2 class distribution")
plt.tight_layout()
plt.savefig(REPORT_DIR / "class_distribution.png", dpi=200, bbox_inches="tight")
plt.show()
print("PASS: exact 63-class order and 8,380-image split verified.")
display(split_summary)
display(class_distribution.sort_values("train_boxes").head(20))

## Cell 9 - Measure Version 1 on Version 2 validation

This creates the fair baseline before fine-tuning. It does not use the test result.

In [ ]:
import json
import numpy as np

baseline_metrics = base_model.val(
    data=str(DATA_YAML), split="val", imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
    device=0, project=str(EVAL_ROOT),
    name=f"v1_model_on_rf_v{VERSION}_validation", exist_ok=True, plots=True,
)
baseline_summary = {
    "model": "Version 1 best.pt",
    "precision": float(baseline_metrics.box.mp),
    "recall": float(baseline_metrics.box.mr),
    "mAP50": float(baseline_metrics.box.map50),
    "mAP50-95": float(baseline_metrics.box.map),
}
(REPORT_DIR / "v1_validation_summary.json").write_text(
    json.dumps(baseline_summary, indent=2), encoding="utf-8"
)
pd.DataFrame({
    "class_id": range(len(EXPECTED_CLASSES)),
    "class_name": EXPECTED_CLASSES,
    "v1_mAP50-95": np.asarray(baseline_metrics.box.maps, dtype=float),
}).to_csv(REPORT_DIR / "v1_validation_per_class.csv", index=False)
display(pd.DataFrame([baseline_summary]))

## Cell 10 - Fine-tune YOLO26s from Version 1 best.pt

This is the long cell. It starts a new run from Version 1 weights with a lower learning rate. Left/right flips are disabled because they would change sign meaning.

In [ ]:
assert not RUN_DIR.exists(), (
    f"Run folder exists: {RUN_DIR}. Use the recovery cell if it was interrupted."
)
finetune_model = YOLO(str(BASE_MODEL_PATH), task="detect")
finetune_model.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
    patience=PATIENCE, device=0, workers=WORKERS,
    project=str(RUN_ROOT), name=RUN_NAME, exist_ok=False,
    pretrained=True, resume=False, optimizer="AdamW",
    lr0=0.001, lrf=0.01, cos_lr=True, weight_decay=0.0005,
    cls_pw=CLASS_WEIGHT_POWER, cache="disk", amp=True, plots=True,
    save_period=5, seed=42, deterministic=True,
    hsv_h=0.015, hsv_s=0.50, hsv_v=0.30,
    degrees=8, translate=0.05, scale=0.25,
    fliplr=0.0, flipud=0.0, mosaic=0.50, close_mosaic=10,
)
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"
assert BEST_PT.is_file(), f"Training did not produce {BEST_PT}"
print("Version 2 best checkpoint:", BEST_PT)

## Cell 11 - Validate Version 2 and compare with Version 1

This saves overall metrics, per-class changes, confusion matrices, prediction examples, and a report-ready comparison chart.

In [ ]:
import json
import numpy as np

BEST_PT = RUN_DIR / "weights" / "best.pt"
baseline_json = REPORT_DIR / "v1_validation_summary.json"
baseline_class_csv = REPORT_DIR / "v1_validation_per_class.csv"
assert BEST_PT.is_file(), f"Missing checkpoint: {BEST_PT}"
assert baseline_json.is_file() and baseline_class_csv.is_file(), "Run Cell 9 first."

best_model = YOLO(str(BEST_PT), task="detect")
v2_metrics = best_model.val(
    data=str(DATA_YAML), split="val", imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
    device=0, project=str(EVAL_ROOT),
    name=f"v2_model_rf_v{VERSION}_validation", exist_ok=True, plots=True,
)
baseline_summary = json.loads(baseline_json.read_text(encoding="utf-8"))
v2_summary = {
    "model": "Version 2 fine-tuned best.pt",
    "precision": float(v2_metrics.box.mp),
    "recall": float(v2_metrics.box.mr),
    "mAP50": float(v2_metrics.box.map50),
    "mAP50-95": float(v2_metrics.box.map),
}
comparison = pd.DataFrame([baseline_summary, v2_summary])
comparison.to_csv(REPORT_DIR / "validation_comparison.csv", index=False)
per_class = pd.read_csv(baseline_class_csv)
per_class["v2_mAP50-95"] = np.asarray(v2_metrics.box.maps, dtype=float)
per_class["change"] = per_class["v2_mAP50-95"] - per_class["v1_mAP50-95"]
per_class.to_csv(REPORT_DIR / "validation_per_class_comparison.csv", index=False)
ax = comparison.set_index("model")[["precision", "recall", "mAP50", "mAP50-95"]].T.plot(
    kind="bar", figsize=(11, 6), color=["#94a3b8", "#2563eb"]
)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("MYSignVoice Version 1 vs Version 2 validation")
ax.legend(loc="lower right")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(REPORT_DIR / "validation_comparison.png", dpi=200, bbox_inches="tight")
plt.show()
display(comparison)
print("Largest per-class decreases:")
display(per_class.sort_values("change").head(15))

## Cell 12 - Optional final test comparison

Keep RUN_FINAL_TEST False while tuning remains possible. Change it to True only after accepting Version 2 using validation.

In [ ]:
RUN_FINAL_TEST = False
if not RUN_FINAL_TEST:
    print("Final test skipped. This is correct while tuning is still possible.")
else:
    base_test = YOLO(str(BASE_MODEL_PATH), task="detect").val(
        data=str(DATA_YAML), split="test", imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
        device=0, project=str(EVAL_ROOT), name=f"v1_on_rf_v{VERSION}_final_test",
        exist_ok=True, plots=True,
    )
    v2_test = YOLO(str(BEST_PT), task="detect").val(
        data=str(DATA_YAML), split="test", imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
        device=0, project=str(EVAL_ROOT), name=f"v2_rf_v{VERSION}_final_test",
        exist_ok=True, plots=True,
    )
    final_test_comparison = pd.DataFrame([
        {"model": "Version 1", "precision": float(base_test.box.mp),
         "recall": float(base_test.box.mr), "mAP50": float(base_test.box.map50),
         "mAP50-95": float(base_test.box.map)},
        {"model": "Version 2", "precision": float(v2_test.box.mp),
         "recall": float(v2_test.box.mr), "mAP50": float(v2_test.box.map50),
         "mAP50-95": float(v2_test.box.map)},
    ])
    final_test_comparison.to_csv(REPORT_DIR / "final_test_comparison.csv", index=False)
    display(final_test_comparison)

## Cell 13 - Export and package the accepted model

Exports ONNX and OpenVINO and creates a timestamped ZIP with the full run, evaluations, report artifacts, data.yaml, and deployment models.

In [ ]:
from datetime import datetime
import json
import shutil

BEST_PT = RUN_DIR / "weights" / "best.pt"
assert BEST_PT.is_file(), f"Missing checkpoint: {BEST_PT}"
best_model = YOLO(str(BEST_PT), task="detect")
onnx_path = Path(best_model.export(format="onnx", imgsz=IMAGE_SIZE, simplify=True, end2end=True))
openvino_path = Path(best_model.export(format="openvino", imgsz=IMAGE_SIZE, end2end=True))
assert onnx_path.is_file(), f"ONNX export missing: {onnx_path}"
assert openvino_path.is_dir(), f"OpenVINO export missing: {openvino_path}"

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
PACKAGE_DIR = BACKUP_ROOT / f"{RUN_NAME}_{stamp}"
PACKAGE_DIR.mkdir(parents=True, exist_ok=False)
shutil.copytree(RUN_DIR, PACKAGE_DIR / "training_run")
shutil.copytree(REPORT_DIR, PACKAGE_DIR / "report_artifacts")
shutil.copytree(EVAL_ROOT, PACKAGE_DIR / "evaluations")
shutil.copy2(DATA_YAML, PACKAGE_DIR / "data.yaml")
manifest = {
    "project": "MYSignVoice", "roboflow_version": VERSION,
    "source_checkpoint": str(BASE_MODEL_PATH),
    "best_checkpoint": "training_run/weights/best.pt",
    "classes": EXPECTED_CLASSES, "image_size": IMAGE_SIZE,
    "split_images": EXPECTED_SPLIT_IMAGES,
}
(PACKAGE_DIR / "model_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
LATEST_ARCHIVE = Path(shutil.make_archive(str(PACKAGE_DIR), "zip", root_dir=PACKAGE_DIR))
print("Package folder:", PACKAGE_DIR)
print("Download before terminating the Pod:", LATEST_ARCHIVE)
print("Version 2 best.pt:", PACKAGE_DIR / "training_run" / "weights" / "best.pt")

## Cell 14 - Clickable ZIP download link

In [ ]:
from IPython.display import FileLink, display
archives = sorted(BACKUP_ROOT.glob(f"{RUN_NAME}_*.zip"), key=lambda p: p.stat().st_mtime)
assert archives, "No ZIP found. Run Cell 13 first."
LATEST_ARCHIVE = archives[-1]
print("ZIP size (GB):", round(LATEST_ARCHIVE.stat().st_size / 1024**3, 3))
display(FileLink(str(LATEST_ARCHIVE)))

## Optional recovery - only after interrupted Cell 10

Use only when the same Version 2 run stopped early and last.pt still exists in /workspace. Do not use it for a different dataset version.

In [ ]:
LAST_PT = RUN_DIR / "weights" / "last.pt"
# assert LAST_PT.is_file(), f"No resumable checkpoint at {LAST_PT}"
# YOLO(str(LAST_PT), task="detect").train(resume=True)